# Multi-Agent News Brief

Find news, filter it with an editor, write a script, and generate a voice briefing with OpenAI or Gemini.

## Workflow overview

This workflow searches for recent news, uses two agents to refine it into a script, then creates an audio briefing. Run the next cell to render the diagram.

In [66]:
%%html
<script src="https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js"></script>

<div class="mermaid">
flowchart TD
    A[Choose a news topic] --> B[DuckDuckGo news search]
    B --> C[News Editor agent: filter and refine]
    C --> D[News Script Writer agent]
    D --> E{Selected provider}
    E -->|OpenAI| F[OpenAI TTS: MP3]
    E -->|Gemini| G[Gemini TTS: WAV]
    F --> H[Play and download audio]
    G --> H
</div>

<script>
  mermaid.initialize({ startOnLoad: false, theme: 'neutral' });
  mermaid.run({ querySelector: '.mermaid' });
</script>

## 1. Install dependencies

In [67]:
%pip install -q openai-agents ddgs google-genai

## 2. Select an agent provider

Set `PROVIDER` to `"openai"` or `"gemini"`. The same provider also generates the final voice briefing.

In [72]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "gemini"  # Change to "openai" to use OpenAI for agents and TTS.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-3.6-flash"

if PROVIDER == "openai":
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GeminiAPIKeysl")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets for Gemini agents.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Find recent news

In [73]:
from ddgs import DDGS


def search_news(query: str) -> str:
    """Find recent news and return each item's title, summary, date, and URL."""
    results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    if not results:
        raise RuntimeError("No recent news results were found. Try a different topic.")

    for index, item in enumerate(results, start=1):
        print(f"{index}. {item.get('title', 'Untitled')}")
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\nDate: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\nURL: {item.get('url', '')}"
        for item in results
    )


topic = "artificial intelligence business"
news_items = search_news(topic)

1. The Benefits And Risks Every Business Leader Needs To Understand About AI
https://www.forbes.com/sites/chuckbrooks/2026/09/12/the-benefits-and-risks-every-business-leader-needs-to-understand-about-ai/
2. Corporate artificial intelligence bills rise despite falling token prices
https://cyprus-mail.com/2026/09/13/corporate-artificial-intelligence-bills-rise-despite-falling-token-prices
3. 'Hit brakes, ban superintelligence': US Senator warns Trump, Xi on AI race after alarm by top industry leaders
https://www.msn.com/en-in/news/world/hit-brakes-ban-superintelligence-us-senator-warns-trump-xi-on-ai-race-after-alarm-by-top-industry-leaders/ar-AA2c6Jc2
4. Bernie Sanders calls for AI pause, ban on superintelligence after Amodei, Musk and Altman back slowdown
https://www.moneycontrol.com/artificial-intelligence/bernie-sanders-calls-for-ai-pause-ban-on-superintelligence-after-amodei-musk-and-altman-back-slowdown-article-14028866.html
5. BigBear.ai vs. IonQ: Weighing Whether to Invest in the

## 4. Editor Agent: filter and refine

In [74]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="Select the three most relevant, credible, non-duplicative items. Exclude ads, clickbait, speculation, and weak evidence. Preserve each selected item's key facts, date, and URL.",
    model=model,
)

editor_result = await Runner.run(editor_agent, f"Topic: {topic}\n\nNews items:\n{news_items}")
edited_news = editor_result.final_output
print(edited_news)

Here are the three most relevant, credible, and non-duplicative news items on artificial intelligence in business:

### 1. Corporate artificial intelligence bills rise despite falling token prices
* **Date:** 2026-09-12T20:01:00+00:00
* **Key Facts:** According to professional services firm PwC, corporate spending on artificial intelligence continues to climb even as raw token costs decrease. PwC notes that standardized cost-control tools are becoming widespread, offering a strategic competitive advantage to businesses that effectively manage rising AI operational expenses as models become commoditized.
* **URL:** https://cyprus-mail.com/2026/09/13/corporate-artificial-intelligence-bills-rise-despite-falling-token-prices

---

### 2. Here's how some Chicago small businesses are ramping up their use of AI
* **Date:** 2026-09-12T11:00:00+00:00
* **Key Facts:** Small businesses are accelerating their integration of practical AI tools like Google's Gemini to streamline everyday operations.

## 5. Writer Agent: create the script

In [75]:
writer_agent = Agent(
    name="News Script Writer",
    instructions="Write a neutral 45- to 60-second spoken news script using only the editor's selected items. Do not add unsupported facts. End by naming the source publications without reading URLs aloud in sinhala language.",
    model=model,
)

writer_result = await Runner.run(writer_agent, f"Create a spoken news script from this edited brief:\n\n{edited_news}")
news_script = writer_result.final_output
print(news_script)

ව්‍යාපාර ක්ෂේත්‍රයේ කෘතිම බුද්ධි තාක්ෂණය භාවිතය පිළිබඳ නවතම පුවත්.

PwC වෘත්තීය සේවා ආයතනයට අනුව, ටෝකන් සඳහා වන මූලික පිරිවැය පහළ ගියද, ව්‍යාපාරික ආයතන කෘතිම බුද්ධිය සඳහා කරන සමස්ත වියදම් තවදුරටත් ඉහළ යමින් පවතිනවා. පිරිවැය පාලන මෙවලම් සාර්ථකව භාවිත කරන ව්‍යාපාරවලට උපායමාර්ගික තරඟකාරී වාසියක් හිමිවන බව ඔවුන් පෙන්වා දෙනවා.

මේ අතර, චිකාගෝ හි සුළු ව්‍යාපාර දෛනික කටයුතු පහසු කරගැනීමට Google Gemini වැනි ප්‍රායෝගික AI මෙවලම් භාවිතය වේගවත් කර තිබෙනවා. Goldman Sachs හි '10,000 Small Businesses' වැඩසටහන මඟින් සපයන අධ්‍යාපනික සම්පත් සහ සම්මන්ත්‍රණ ඊට සහාය දක්වනවා.

තවද, මූල්‍ය විශ්ලේෂණවලට අනුව BigBear.ai සමාගම ආදායම් පහත වැටීම් සහ ගිණුම්කරණ සංශෝධනවලට මුහුණ දී සිටින අතර, ක්වොන්ටම් පරිගණක සමාගමක් වන IonQ ඉහළ මුදල් වැයවීම් සමඟ තෙඉලක්කම් වර්ධනයක් වාර්තා කර තිබෙනවා.

මෙම පුවත් Cyprus Mail, Chicago Sun-Times සහ Yahoo Finance යන ප්‍රකාශන මගින් වාර්තා කර තිබුණා.


## 6. Generate the voice briefing

This creates AI-generated speech with the selected provider. Disclose that the voice is AI-generated when sharing it.

In [76]:
from IPython.display import Audio, display

if PROVIDER == "openai":
    tts_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    speech = await tts_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input=news_script,
        instructions="Speak clearly in a calm, neutral broadcast-news style in sinhala language.",
    )
    audio_path = "news_brief.mp3"
    speech.write_to_file(audio_path)
else:
    import base64
    import wave

    from google import genai

    def save_wav(filename, pcm, channels=1, rate=24000, sample_width=2):
        with wave.open(filename, "wb") as wav_file:
            wav_file.setnchannels(channels)
            wav_file.setsampwidth(sample_width)
            wav_file.setframerate(rate)
            wav_file.writeframes(pcm)

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    response = gemini_client.interactions.create(
        model="gemini-3.1-flash-tts-preview",
        input=(
            "Read this in a calm, neutral broadcast-news style:\n\n"
            f"{news_script}"
        ),
        response_format={"type": "audio"},
        generation_config={"speech_config": [{"voice": "Kore"}]},
    )
    audio_path = "news_brief.wav"
    save_wav(audio_path, base64.b64decode(response.output_audio.data))

display(Audio(audio_path))


## 7. Download the audio file

Run this cell to save the generated MP3 (OpenAI) or WAV (Gemini) file to your computer.

In [ ]:
from google.colab import files

files.download(audio_path)